In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!unzip /content/drive/My\ Drive/cards.zip

Streaming output truncated to the last 5000 lines.
  inflating: cards/train/queen of spades/066.jpg  
  inflating: cards/train/queen of spades/099.jpg  
  inflating: cards/train/queen of spades/106.jpg  
  inflating: cards/train/queen of spades/112.jpg  
  inflating: cards/train/queen of spades/104.jpg  
  inflating: cards/train/queen of spades/110.jpg  
  inflating: cards/train/queen of spades/138.jpg  
  inflating: cards/train/queen of spades/070.jpg  
  inflating: cards/train/queen of spades/064.jpg  
  inflating: cards/train/queen of spades/058.jpg  
  inflating: cards/train/queen of spades/059.jpg  
  inflating: cards/train/queen of spades/065.jpg  
  inflating: cards/train/queen of spades/071.jpg  
  inflating: cards/train/queen of spades/139.jpg  
  inflating: cards/train/queen of spades/111.jpg  
  inflating: cards/train/queen of spades/105.jpg  
  inflating: cards/train/queen of spades/108.jpg  
  inflating: cards/train/queen of spades/120.jpg  
  inflating: cards/train/queen 

In [2]:
import keras
keras.__version__

'3.13.2'

In [3]:
import os, shutil

In [4]:
base_dir = '/content//cards'

train_dir = os.path.join(base_dir, 'train')
validation_dir = os.path.join(base_dir, 'valid')
test_dir = os.path.join(base_dir, 'test')


In [5]:
from tensorflow.keras import layers
from tensorflow.keras.utils import image_dataset_from_directory

train_dataset = image_dataset_from_directory(
    train_dir,
    image_size=(160, 160),
    batch_size=32
)
validation_dataset = image_dataset_from_directory(
    validation_dir,
    image_size=(160, 160),
    batch_size=32
)
test_dataset = image_dataset_from_directory(
    test_dir,
    image_size=(160, 160),
    batch_size=32
)

class_names = train_dataset.class_names
num_classes = len(class_names)

Found 7509 files belonging to 52 classes.
Found 260 files belonging to 52 classes.
Found 260 files belonging to 52 classes.


In [6]:
import tensorflow as tf

AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
validation_dataset = validation_dataset.cache().prefetch(buffer_size=AUTOTUNE)
test_dataset = test_dataset.cache().prefetch(buffer_size=AUTOTUNE)

In [7]:
from tensorflow.keras import layers, models, regularizers

data_augmentation = tf.keras.Sequential([
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.08, 0.08),
    layers.RandomContrast(0.1),
])

model = models.Sequential([
    layers.Input(shape=(160, 160, 3)),
    data_augmentation,
    layers.Rescaling(1.0 / 255),
    # Block 1
    layers.Conv2D(32, (3, 3), padding="same", use_bias=False),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Conv2D(32, (3, 3), padding="same", use_bias=False),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.15),
    # Block 2
    layers.Conv2D(64, (3, 3), padding="same", use_bias=False),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Conv2D(64, (3, 3), padding="same", use_bias=False),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.2),
    # Block 3
    layers.Conv2D(128, (3, 3), padding="same", use_bias=False),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Conv2D(128, (3, 3), padding="same", use_bias=False),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),
    # Block 4
    layers.Conv2D(256, (3, 3), padding="same", use_bias=False),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Conv2D(256, (3, 3), padding="same", use_bias=False),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.3),
    # Classifier Head
    layers.GlobalAveragePooling2D(),
    layers.Dense(
        256,
        use_bias=False,
        kernel_regularizer=regularizers.l2(1e-4),
    ),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.4),
    layers.Dense(num_classes, activation="softmax"),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential (Sequential)         │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 160, 160, 32)   │           864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 160, 160, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 160, 160, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 160, 160, 32)   │         9,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 160, 160, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 160, 160, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 80, 80, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 80, 80, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 80, 80, 64)     │        18,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 80, 80, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 80, 80, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 80, 80, 64)     │        36,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 80, 80, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 80, 80, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 40, 40, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 40, 40, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 40, 40, 128)    │        73,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 40, 40, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_4 (Activation)       │ (None, 40, 40, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 40, 40, 128)    │       147,456 │
├─────────────────────────────────┼────────────────────────┼─────────────

 Total params: 1,255,060 (4.79 MB)

 Trainable params: 1,252,628 (4.78 MB)

 Non-trainable params: 2,432 (9.50 KB)

In [8]:
callbacks = [
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy", patience=7, restore_best_weights=True, verbose=1
    ),
]

history = model.fit(
    train_dataset, validation_data=validation_dataset, epochs=35, callbacks=callbacks
)

Epoch 1/35
235/235 ━━━━━━━━━━━━━━━━━━━━ 68s 198ms/step - accuracy: 0.0790 - loss: 3.5725 - val_accuracy: 0.0346 - val_loss: 5.1006 - learning_rate: 0.0010
Epoch 2/35
235/235 ━━━━━━━━━━━━━━━━━━━━ 38s 163ms/step - accuracy: 0.1212 - loss: 3.1074 - val_accuracy: 0.1731 - val_loss: 2.5925 - learning_rate: 0.0010
Epoch 3/35
235/235 ━━━━━━━━━━━━━━━━━━━━ 40s 172ms/step - accuracy: 0.1722 - loss: 2.8469 - val_accuracy: 0.1500 - val_loss: 2.8898 - learning_rate: 0.0010
Epoch 4/35
235/235 ━━━━━━━━━━━━━━━━━━━━ 39s 166ms/step - accuracy: 0.2279 - loss: 2.6221 - val_accuracy: 0.0923 - val_loss: 4.8862 - learning_rate: 0.0010
Epoch 5/35
235/235 ━━━━━━━━━━━━━━━━━━━━ 40s 171ms/step - accuracy: 0.2710 - loss: 2.4185 - val_accuracy: 0.3346 - val_loss: 2.0075 - learning_rate: 0.0010
Epoch 6/35
235/235 ━━━━━━━━━━━━━━━━━━━━ 40s 168ms/step - accuracy: 0.3281 - loss: 2.2603 - val_accuracy: 0.2154 - val_loss: 2.7680 - learning_rate: 0.0010
Epoch 7/35
235/235 ━━━━━━━━━━━━━━━━━━━━ 41s 169ms/step - accuracy: 0.3

In [11]:
test_loss, test_acc = model.evaluate(test_dataset)
test_acc_percent = test_acc * 100

print(f"\nTest Accuracy: {test_acc_percent:.2f}%")

9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.8192 - loss: 0.6257

Test Accuracy: 81.92%
